# 09 — LLM Logic Benchmark

**Question**: Do LLMs reason classically, or do they implicitly use a non-classical logic?

Most LLM-reasoning evals assume classical first-order logic as ground truth. But classical logic is just one of many. Real reasoning under uncertainty, contradiction, or relevance constraints often diverges from classical entailment.

This notebook builds a benchmark of natural-language entailment problems whose **correct answer differs** depending on which logic you adopt:

| Logic | Key feature | Diverges from classical when... |
|-------|-------------|---------------------------------|
| Classical (CL) | Bivalent, ex falso quodlibet | (baseline) |
| Intuitionistic (IL) | Constructive: rejects LEM, double-negation elimination | proof requires construction, ¬¬φ ⊬ φ |
| Paraconsistent (LP) | Contradiction does not explode | premises contradictory but conclusion irrelevant |
| Relevance (R) | Premise must be *relevant* to conclusion | A ∧ ¬A ⊬ B (irrelevance) |

Each item has 4 expected answers (one per logic). We then ask LLMs and see which logic's pattern they match.

In [1]:
from dataclasses import dataclass, field, asdict
from typing import Literal
import json

Verdict = Literal["entails", "does_not_entail"]
LogicName = Literal["classical", "intuitionistic", "paraconsistent", "relevance"]

@dataclass
class BenchmarkItem:
    id: str
    category: str             # 'LEM', 'EFQ', 'DNE', 'relevance', 'constructive_witness'
    premises: list[str]       # natural-language premises
    conclusion: str           # natural-language conclusion
    formal: str               # symbolic form (for the appendix)
    expected: dict[LogicName, Verdict]  # answer per logic
    explanation: str          # one-line why each logic differs

## Benchmark items

Each item is hand-crafted so that classical + intuitionistic + paraconsistent + relevance logics give at least 2 distinct verdicts.

In [2]:
BENCHMARK: list[BenchmarkItem] = [
    # ---- Law of Excluded Middle (CL=yes, IL=no) ----
    BenchmarkItem(
        id="LEM-01",
        category="LEM",
        premises=["There exists either a largest twin prime or there does not."],
        conclusion="The disjunction 'there is a largest twin prime, or there is not' is true.",
        formal="⊢ φ ∨ ¬φ",
        expected={"classical": "entails", "intuitionistic": "does_not_entail",
                  "paraconsistent": "entails", "relevance": "entails"},
        explanation="Intuitionism rejects LEM without a constructive witness.",
    ),
    BenchmarkItem(
        id="LEM-02",
        category="LEM",
        premises=["Either Goldbach's conjecture is true, or it is false."],
        conclusion="This is a logical truth.",
        formal="⊢ φ ∨ ¬φ",
        expected={"classical": "entails", "intuitionistic": "does_not_entail",
                  "paraconsistent": "entails", "relevance": "entails"},
        explanation="Same LEM pattern, different framing.",
    ),
    # ---- Double Negation Elimination (CL=yes, IL=no) ----
    BenchmarkItem(
        id="DNE-01",
        category="DNE",
        premises=["It is not the case that the function f has no zero."],
        conclusion="f has a zero.",
        formal="¬¬φ ⊢ φ",
        expected={"classical": "entails", "intuitionistic": "does_not_entail",
                  "paraconsistent": "entails", "relevance": "entails"},
        explanation="Intuitionism: ¬¬φ doesn't constructively give φ.",
    ),
    BenchmarkItem(
        id="DNE-02",
        category="DNE",
        premises=["It is not the case that no winning strategy exists for player A."],
        conclusion="Player A has a winning strategy.",
        formal="¬¬∃x.W(x) ⊢ ∃x.W(x)",
        expected={"classical": "entails", "intuitionistic": "does_not_entail",
                  "paraconsistent": "entails", "relevance": "entails"},
        explanation="Constructive existence requires a witness.",
    ),
    # ---- Ex Falso Quodlibet (CL=yes, LP=no, R=no) ----
    BenchmarkItem(
        id="EFQ-01",
        category="EFQ",
        premises=["The Earth is round.", "The Earth is not round."],
        conclusion="The moon is made of cheese.",
        formal="φ, ¬φ ⊢ ψ",
        expected={"classical": "entails", "intuitionistic": "entails",
                  "paraconsistent": "does_not_entail", "relevance": "does_not_entail"},
        explanation="Paraconsistent and relevance logics block explosion.",
    ),
    BenchmarkItem(
        id="EFQ-02",
        category="EFQ",
        premises=["All swans are white.", "There exists a black swan."],
        conclusion="All cars are blue.",
        formal="φ, ¬φ ⊢ ψ",
        expected={"classical": "entails", "intuitionistic": "entails",
                  "paraconsistent": "does_not_entail", "relevance": "does_not_entail"},
        explanation="Contradiction about swans does not bear on cars.",
    ),
    # ---- Relevance violation (CL=yes, R=no) ----
    BenchmarkItem(
        id="REL-01",
        category="relevance",
        premises=["It is raining."],
        conclusion="If 2+2=5, then it is raining.",
        formal="φ ⊢ ψ → φ",
        expected={"classical": "entails", "intuitionistic": "entails",
                  "paraconsistent": "entails", "relevance": "does_not_entail"},
        explanation="Relevance: antecedent must be relevant to consequent.",
    ),
    BenchmarkItem(
        id="REL-02",
        category="relevance",
        premises=["Paris is the capital of France."],
        conclusion="If the moon is square, then Paris is the capital of France.",
        formal="φ ⊢ ψ → φ",
        expected={"classical": "entails", "intuitionistic": "entails",
                  "paraconsistent": "entails", "relevance": "does_not_entail"},
        explanation="Vacuous conditional only valid in classical/intuitionistic.",
    ),
    # ---- Disjunctive syllogism — kills LP and weak relevance ----
    BenchmarkItem(
        id="DS-01",
        category="disjunctive_syllogism",
        premises=["Either Anna or Bob committed the crime.", "Anna did not commit the crime."],
        conclusion="Bob committed the crime.",
        formal="φ ∨ ψ, ¬φ ⊢ ψ",
        expected={"classical": "entails", "intuitionistic": "entails",
                  "paraconsistent": "does_not_entail", "relevance": "does_not_entail"},
        explanation="In LP, ¬φ alone doesn't eliminate φ from the disjunction.",
    ),
    # ---- Constructive existence ----
    BenchmarkItem(
        id="CON-01",
        category="constructive_witness",
        premises=["It would be contradictory if no real number squared to 2."],
        conclusion="There is a real number whose square is 2.",
        formal="¬¬∃x.φ(x) ⊢ ∃x.φ(x)",
        expected={"classical": "entails", "intuitionistic": "does_not_entail",
                  "paraconsistent": "entails", "relevance": "entails"},
        explanation="Classical proof of existence by contradiction; not constructive.",
    ),
    # ---- Peirce's law (CL=yes, IL=no) ----
    BenchmarkItem(
        id="PEI-01",
        category="Peirce",
        premises=["If 'if it rains then I get wet' implies it rains, then it rains."],
        conclusion="This is a logical tautology.",
        formal="⊢ ((φ → ψ) → φ) → φ",
        expected={"classical": "entails", "intuitionistic": "does_not_entail",
                  "paraconsistent": "entails", "relevance": "does_not_entail"},
        explanation="Peirce's law fails in IL and most relevance logics.",
    ),
    # ---- Material implication oddity ----
    BenchmarkItem(
        id="MAT-01",
        category="material_implication",
        premises=["If the moon is made of cheese, then 2+2=4."],
        conclusion="This conditional is true.",
        formal="⊢ φ → ψ where ¬φ",
        expected={"classical": "entails", "intuitionistic": "entails",
                  "paraconsistent": "entails", "relevance": "does_not_entail"},
        explanation="Vacuous truth from false antecedent; relevance rejects.",
    ),
]

print(f"Loaded {len(BENCHMARK)} benchmark items")
print(f"Categories: {sorted(set(item.category for item in BENCHMARK))}")

Loaded 12 benchmark items
Categories: ['DNE', 'EFQ', 'LEM', 'Peirce', 'constructive_witness', 'disjunctive_syllogism', 'material_implication', 'relevance']


## Logic-divergence summary

Items where all 4 logics agree are useless — drop them. Items where ≥2 logics diverge are signal.

In [3]:
from collections import Counter

def divergence_count(item: BenchmarkItem) -> int:
    return len(set(item.expected.values()))

for item in BENCHMARK:
    div = divergence_count(item)
    flag = "✓ signal" if div >= 2 else "✗ no signal"
    print(f"{item.id} ({item.category:25s}) divergence={div} {flag}")

category_counts = Counter(item.category for item in BENCHMARK)
print(f"\nCategory distribution: {dict(category_counts)}")

LEM-01 (LEM                      ) divergence=2 ✓ signal
LEM-02 (LEM                      ) divergence=2 ✓ signal
DNE-01 (DNE                      ) divergence=2 ✓ signal
DNE-02 (DNE                      ) divergence=2 ✓ signal
EFQ-01 (EFQ                      ) divergence=2 ✓ signal
EFQ-02 (EFQ                      ) divergence=2 ✓ signal
REL-01 (relevance                ) divergence=2 ✓ signal
REL-02 (relevance                ) divergence=2 ✓ signal
DS-01 (disjunctive_syllogism    ) divergence=2 ✓ signal
CON-01 (constructive_witness     ) divergence=2 ✓ signal
PEI-01 (Peirce                   ) divergence=2 ✓ signal
MAT-01 (material_implication     ) divergence=2 ✓ signal

Category distribution: {'LEM': 2, 'DNE': 2, 'EFQ': 2, 'relevance': 2, 'disjunctive_syllogism': 1, 'constructive_witness': 1, 'Peirce': 1, 'material_implication': 1}


## Save benchmark as JSON for use in notebook 10

In [4]:
import os
os.makedirs("benchmark_data", exist_ok=True)

with open("benchmark_data/llm_logic_benchmark.json", "w", encoding="utf-8") as f:
    json.dump([asdict(item) for item in BENCHMARK], f, indent=2, ensure_ascii=False)

print(f"Saved {len(BENCHMARK)} items to benchmark_data/llm_logic_benchmark.json")

Saved 12 items to benchmark_data/llm_logic_benchmark.json


## Next

→ Notebook 10: feed each item to Claude / Llama via the Router pattern from `Claude assistant/multiagent_tool.py`, collect verdict + reasoning trace.

→ Notebook 11: aggregate per-model verdicts, compute distance to each logic's expected answer pattern → produce **logic fingerprint**.